# 02. LLM Basics for Causal Analysts

Large language models can help causal analysts draft, critique, and organize causal work. But they are not causal identification engines. They are text-generation systems that respond to prompts, context, examples, model scale, decoding choices, and training data.

This notebook gives the minimum LLM literacy needed for the rest of the **AI for Causal Inference** course. We will focus on the pieces that matter for causal workflows: prompts, chat templates, model scale, temperature, structured outputs, hallucination, reproducibility, and human review.

## Learning Goals

By the end of this notebook, you should be able to:

- Explain the difference between a pretrained model, an instruction-tuned model, and a chat interface.
- Use system and user messages to constrain an LLM's role in a causal workflow.
- Understand why chat templates matter for local models.
- Use deterministic settings for reproducible causal-analysis artifacts.
- Generate and validate structured outputs with `pydantic`.
- Diagnose common LLM failure modes in causal work: hallucination, overclaiming, missing assumptions, and invalid adjustment recommendations.
- Design a model-scale and model-family comparison that evaluates causal quality rather than vibes.

## Live Model Note

This course treats LLM behavior as an empirical object. These notebooks may include live local-model calls, so outputs can vary across model versions, hardware, decoding settings, prompt wording, package versions, and reruns. That instability is part of the lesson: AI-assisted causal inference requires validation, audit trails, and analyst judgment.

Treat model output as a draft artifact, not as causal evidence. A model may produce valid JSON with weak causal reasoning, or strong prose that fails schema validation.

When live calls are enabled, read the results as experiments about AI behavior:

- Did the model invent design details?
- Did it confuse prediction with causation?
- Did it recommend bad controls?
- Did it obey the schema?
- Did it surface missing information?
- Did it preserve uncertainty?

The goal is not to make every model output perfect. The goal is to learn how to build AI-assisted causal workflows that are auditable, constrained, and reviewed by a human analyst.

## 1. Why Causal Analysts Need LLM Literacy

In ordinary analytics, a sloppy LLM answer may be merely annoying. In causal inference, a sloppy LLM answer can be actively harmful because the model may:

- turn an association into a causal claim;
- recommend controlling for a mediator or collider;
- invent a randomized experiment that did not happen;
- ignore timing and post-treatment variables;
- choose a method because a keyword appears in the prompt;
- produce a polished executive memo that hides weak identification.

The right posture is not distrust for its own sake. The right posture is disciplined use: make the model produce artifacts that are easy to inspect, validate, and revise.

## 2. Setup

This notebook can run in two modes:

- **fallback mode**, which uses deterministic example outputs and is safe for public rendering;
- **live local LLM mode**, which uses a local Hugging Face model through `transformers`.

The default is fallback mode. To run a local model interactively, set `RUN_LIVE_LOCAL_LLM = True`.

In [37]:
import gc
import importlib.util
import json
import os
import re
import textwrap
from functools import lru_cache

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from pydantic import BaseModel, Field, ValidationError

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

In [38]:
RUN_LIVE_LOCAL_LLM = True

LOCAL_SMOKE_TEST_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
LOCAL_FAST_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LOCAL_STRONG_MODEL = 'Qwen/Qwen2.5-14B-Instruct'
LOCAL_SCALE_MODEL = 'Qwen/Qwen2.5-32B-Instruct'

LOCAL_ALT_REASONING_MODEL = 'microsoft/Phi-3.5-mini-instruct'
LOCAL_ALT_OPEN_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'
LOCAL_MISTRAL_SMALL_MODEL = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'
LOCAL_GEMMA_MODEL = 'google/gemma-3-27b-it'
LOCAL_LLAMA_MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

# For interactive testing, start with the smoke-test model. Use 7B/14B/32B after the pipeline works.
MODEL_ID = LOCAL_SMOKE_TEST_MODEL

# Keep the full benchmark off during website rendering. Turn it on only in an interactive GPU session.
RUN_FULL_MODEL_COMPARISON = True
COMPARISON_MAX_NEW_TOKENS = 360
COMPARISON_TEMPERATURE = 0.0

MODELS_TO_COMPARE = [
    ('Qwen 0.5B', LOCAL_SMOKE_TEST_MODEL, 'pipeline smoke test'),
    ('Qwen 7B', LOCAL_FAST_MODEL, 'fast default'),
    ('Qwen 14B', LOCAL_STRONG_MODEL, 'strong local analysis'),
    ('Qwen 32B', LOCAL_SCALE_MODEL, 'scale comparison'),
    ('Phi mini', LOCAL_ALT_REASONING_MODEL, 'compact non-Qwen comparison'),
    ('Mistral 7B', LOCAL_ALT_OPEN_MODEL, '7B model-family comparison'),
    ('Mistral Small 24B', LOCAL_MISTRAL_SMALL_MODEL, 'strong non-Qwen comparison'),
    ('Gemma 3 27B', LOCAL_GEMMA_MODEL, 'large non-Qwen comparison'),
    ('Llama 3.1 8B', LOCAL_LLAMA_MODEL, 'industry-standard instruct baseline'),
]

MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0
SEED = 123

In [39]:
def has_package(module_name):
    return importlib.util.find_spec(module_name) is not None


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

environment_status = pd.DataFrame(
    [
        ('torch', has_package('torch'), torch.__version__),
        ('transformers', has_package('transformers'), ''),
        ('accelerate', has_package('accelerate'), ''),
        ('pydantic', has_package('pydantic'), ''),
        ('cuda available', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not visible'),
    ],
    columns=['component', 'available', 'details'],
)

environment_status

,component,available,details
0,torch,True,2.11.0+cu130
1,transformers,True,
2,accelerate,True,
3,pydantic,True,
4,cuda available,True,NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition


### Discussion

The notebook is written to use GPU if available. In some sandboxed rendering environments, PyTorch may not see the GPU even when your terminal can. That is fine for the public notebook. When you run locally, the same code will use CUDA if `torch.cuda.is_available()` is `True`.

## 3. Model Ladder for This Course

The course uses a model ladder because we care about both **scale** and **model family**.

Scale lets us ask whether larger models produce better causal artifacts. Model-family comparisons let us ask whether different instruction-tuned systems make different mistakes.

In [40]:
model_ladder = pd.DataFrame(MODELS_TO_COMPARE, columns=['model', 'model_id', 'course use'])

model_ladder

,model,model_id,course use
0,Qwen 0.5B,Qwen/Qwen2.5-0.5B-Instruct,pipeline smoke test
1,Qwen 7B,Qwen/Qwen2.5-7B-Instruct,fast default
2,Qwen 14B,Qwen/Qwen2.5-14B-Instruct,strong local analysis
3,Qwen 32B,Qwen/Qwen2.5-32B-Instruct,scale comparison
4,Phi mini,microsoft/Phi-3.5-mini-instruct,compact non-Qwen comparison
5,Mistral 7B,mistralai/Mistral-7B-Instruct-v0.3,7B model-family comparison
6,Mistral Small 24B,mistralai/Mistral-Small-3.1-24B-Instruct-2503,strong non-Qwen comparison
7,Gemma 3 27B,google/gemma-3-27b-it,large non-Qwen comparison
8,Llama 3.1 8B,meta-llama/Meta-Llama-3.1-8B-Instruct,industry-standard instruct baseline


### Local LLM Smoke Test Report

Before writing this notebook, we tested the real local-generation path with `Qwen/Qwen2.5-0.5B-Instruct`.

- The model downloaded successfully.
- The model loaded successfully with `transformers`.
- A raw prompt produced a weak answer.
- The same task with a proper chat template produced a much better answer.

That result is important for the course: the local LLM stack works, but prompt format and model quality matter. The 0.5B model is useful for testing infrastructure, not for serious causal critique.

## 4. Basic LLM Concepts

For causal analysts, the most important LLM concepts are:

| Concept | Meaning | Why it matters for causal work |
|---|---|---|
| Pretrained model | A model trained to predict text from large corpora | It has broad language knowledge, but not guaranteed truthfulness |
| Instruction-tuned model | A model further trained to follow instructions | Better for analysis tasks, but still not a causal authority |
| Chat template | The formatting convention for system/user/assistant messages | Local chat models often perform badly if you skip it |
| Context window | The text the model can see at generation time | Missing context leads to invented assumptions |
| Temperature | Controls randomness in sampling | Low temperature is preferred for reproducible analysis artifacts |
| Structured output | JSON or schema-constrained response | Makes model outputs easier to validate and test |
| Hallucination | Plausible but unsupported output | Especially dangerous when it invents causal design details |
| Guardrail | A rule, check, or validation step around the model | Helps prevent polished causal mistakes |

## 5. Why Chat Templates Matter

Instruction models are usually trained on a particular conversation format. If we give the model a plain string that says `System: ... User: ... Assistant:`, it may still work, but we are not using the exact format it expects.

With Hugging Face models, the tokenizer often knows the right chat template. The helper below uses `tokenizer.apply_chat_template` when available.

In [41]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebooks._shared.local_llm import (
    DEFAULT_MODELS_TO_COMPARE,
    build_chat_inputs,
    clean_generated_text,
    clear_loaded_model_cache,
    decode_generated_response,
    format_chat_prompt,
    get_device,
    has_package,
    load_local_model as _shared_load_local_model,
    local_chat as _shared_local_chat,
    move_inputs_to_model_device,
    prepare_chat_inputs,
    set_generation_seed,
)

DEVICE = get_device()


def load_local_model(model_id=MODEL_ID):
    return _shared_load_local_model(model_id)


def decode_generated_text(tokenizer, generated, prompt_token_count, model_id=MODEL_ID):
    return decode_generated_response(tokenizer, generated, prompt_token_count, model_id=model_id)


def local_chat(user_message, system_message=None, model_id=MODEL_ID, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    enabled = globals().get('RUN_LIVE_LOCAL_LLM', globals().get('RUN_LOCAL_LLM', True))
    return _shared_local_chat(
        user_message,
        system_message=system_message,
        model_id=model_id,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        seed=globals().get('SEED', 123),
        enabled=enabled,
    )


def local_generate(user_message, system_message=None, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    return _shared_local_chat(
        user_message,
        system_message=system_message,
        model_id=MODEL_ID,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        seed=globals().get('SEED', 123),
        enabled=globals().get('RUN_LOCAL_LLM', globals().get('RUN_LIVE_LOCAL_LLM', True)),
    )


## 6. A Reproducible Wrapper

Public notebooks should not fail when model weights are unavailable. The wrapper below calls the local LLM only if `RUN_LIVE_LOCAL_LLM = True`; otherwise it returns a deterministic fallback example.

This is the pattern we will reuse throughout the course.

In [42]:
FALLBACK_RESPONSES = {
    'naive_comparison': (
        'Naive treated-versus-control comparisons can fail because treatment assignment is often related to baseline risk; '
        'if high-risk units are more likely to receive treatment, the comparison mixes treatment effects with selection bias.'
    ),
    'estimand_json': json.dumps(
        {
            'treatment': 'proactive customer-success outreach',
            'outcome': 'churn within 90 days',
            'unit': 'customer account',
            'time_window': 'eligible accounts in the prior quarter with 90-day follow-up',
            'main_assumptions': [
                'observed pre-treatment covariates capture important targeting logic',
                'no adjustment for post-treatment engagement when estimating the total effect',
                'sufficient overlap between treated and untreated accounts',
            ],
        },
        indent=2,
    ),
    'bad_answer': 'The campaign was randomized, so the higher churn among treated accounts proves the campaign caused churn.',
}


def causal_llm(user_message, prompt_name='naive_comparison', system_message=None, **kwargs):
    if RUN_LIVE_LOCAL_LLM:
        return local_chat(user_message, system_message=system_message, **kwargs)
    return FALLBACK_RESPONSES.get(prompt_name, 'Fallback response: local LLM disabled for reproducible rendering.')


smoke_prompt = 'Why can naive treated-versus-control comparisons fail in observational causal inference?'
print(causal_llm(smoke_prompt, prompt_name='naive_comparison'))

Naive treatment versus control comparisons can fail in observational causal inference due to several reasons:

1. **Heterogeneity of Treatment Effects**: The observed differences between the groups (treatment and control) may be influenced by various factors that differ across the two groups, such as confounding variables or unmeasured mediators.

2. **Measurement Bias**: The measures used for comparison might not accurately reflect the true effects of the treatments. For example, if the outcome measure is not reliable or if it changes over time, this could lead to biased estimates.

3. **Confounding Variables**: Confounders are variables that affect both the exposure and the outcome but cannot be controlled for. They can influence the relationship between the exposure and the outcome, making it difficult to attribute the effect of the treatment to the intervention.

4. **Time-Varying Effects**: If the outcomes change over time, the difference between the treatment group and the contro

### Turning on the Live Local Model

To run the live model interactively:

```python
RUN_LIVE_LOCAL_LLM = True
MODEL_ID = LOCAL_FAST_MODEL      # or LOCAL_STRONG_MODEL / LOCAL_SCALE_MODEL
```

Start with the smoke-test model first. Once that works, move to 7B, 14B, and 32B.

In [43]:
if RUN_LIVE_LOCAL_LLM:
    response = causal_llm(
        (
            'Return exactly three concise bullet points. '
            'Each bullet must be one sentence and no more than 25 words. '
            'Explain why an LLM should not conclude treatment causation from a raw treated-versus-control difference.'
        ),
        model_id=MODEL_ID,
        max_new_tokens=260,
    )
    display(Markdown(response))
else:
    print('RUN_LIVE_LOCAL_LLM is False. Skipping live local LLM call in the rendered notebook.')

- An LLM should not conclusively determine that treatment causes the observed difference if the control group did not receive the intervention.
- The use of a placebo or sham treatment can mask true effects, making it challenging to establish cause-and-effect relationships definitively.
- Raw data without controlling for confounding variables may lead to spurious associations between treatments and outcomes.

## 7. Prompt Anatomy for Causal Work

A causal prompt should usually contain four pieces:

1. **Role:** what the model is supposed to do.
2. **Task:** the artifact to produce.
3. **Context:** project details and available evidence.
4. **Constraints:** what the model must not assume or overstate.

The most important habit is to ask for reviewable artifacts, not conclusions.

In [44]:
prompt_examples = pd.DataFrame(
    [
        (
            'weak',
            'Did outreach reduce churn?',
            'Invites a causal conclusion before defining treatment, outcome, design, or assumptions.',
        ),
        (
            'better',
            'Draft an estimand card for evaluating proactive outreach on 90-day churn. List treatment, outcome, unit, time window, assumptions, and open questions. Do not estimate the effect.',
            'Forces the model to produce a design artifact instead of an answer.',
        ),
        (
            'best',
            'Given the project brief and variable list, classify each variable as pre-treatment confounder, mediator, collider, outcome, or irrelevant. Flag any variable that is unsafe to adjust for when estimating the total effect.',
            'Targets a specific causal reasoning task and makes bad-control detection explicit.',
        ),
    ],
    columns=['prompt quality', 'prompt', 'why it matters'],
)

prompt_examples

,prompt quality,prompt,why it matters
0,weak,Did outreach reduce churn?,"Invites a causal conclusion before defining treatment, outcome, design, or assumptions."
1,better,"Draft an estimand card for evaluating proactive outreach on 90-day churn. List treatment, outcome, unit, time window...",Forces the model to produce a design artifact instead of an answer.
2,best,"Given the project brief and variable list, classify each variable as pre-treatment confounder, mediator, collider, o...",Targets a specific causal reasoning task and makes bad-control detection explicit.


## 8. System Messages

A system message is not magic, but it is useful. It sets the model's role and behavioral constraints.

For causal analysis, a strong system message should tell the model to:

- distinguish association from causation;
- state assumptions explicitly;
- flag missing information;
- avoid inventing study design details;
- avoid recommending post-treatment adjustment for total effects;
- separate evidence from recommendations.

In [45]:
CAUSAL_SYSTEM_MESSAGE = textwrap.dedent(
    '''
    You are a careful causal inference assistant.
    Your job is to draft reviewable causal-analysis artifacts, not to overstate conclusions.
    Always distinguish association from causation.
    State required assumptions and flag missing information.
    Do not invent randomization, instruments, thresholds, or temporal ordering.
    Do not recommend adjusting for post-treatment variables when estimating a total effect.
    '''
).strip()

print(CAUSAL_SYSTEM_MESSAGE)

You are a careful causal inference assistant.
Your job is to draft reviewable causal-analysis artifacts, not to overstate conclusions.
Always distinguish association from causation.
State required assumptions and flag missing information.
Do not invent randomization, instruments, thresholds, or temporal ordering.
Do not recommend adjusting for post-treatment variables when estimating a total effect.


## 9. Structured Outputs and JSON Schemas

Unstructured prose is useful for explanation. Structured outputs are better for workflows.

For example, if the model drafts an estimand card as JSON, we can validate that required fields are present before using it downstream.

In [46]:
class CausalDesignDraft(BaseModel):
    treatment: str
    outcome: str
    unit: str
    time_window: str
    main_assumptions: list[str] = Field(min_length=1)
    missing_information: list[str] = Field(default_factory=list)
    unsafe_adjustment_variables: list[str] = Field(default_factory=list)


JSON_ONLY_SYSTEM_MESSAGE = textwrap.dedent(
    '''
    You are a careful causal inference assistant.
    Return only valid JSON that matches the requested schema.
    Do not include markdown fences, explanations, preambles, or scratch work.
    If information is missing, use missing_information rather than inventing facts.
    '''
).strip()


structured_prompt = textwrap.dedent(
    '''
    Draft a JSON object for evaluating whether proactive outreach reduces churn.
    The JSON object must have exactly these keys:
    treatment, outcome, unit, time_window, main_assumptions,
    missing_information, unsafe_adjustment_variables.

    main_assumptions, missing_information, and unsafe_adjustment_variables must be arrays of strings.
    Return the JSON object only.
    '''
).strip()

raw_structured_output = causal_llm(
    structured_prompt,
    prompt_name='estimand_json',
    system_message=JSON_ONLY_SYSTEM_MESSAGE,
    max_new_tokens=450,
)
print(raw_structured_output)

```json
{
 "treatment": "proactive outreach",
 "outcome": "churn",
 "unit": "customers",
 "time_window": "12 months",
 "main_assumptions": ["proactive outreach leads to reduced churn"],
 "missing_information": [],
 "unsafe_adjustment_variables": []
}
```


In [47]:
def extract_json_object(text):
    cleaned = clean_generated_text(text)

    # Remove common markdown code fences if the model adds them despite instructions.
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned.strip())
    cleaned = re.sub(r'\s*```$', '', cleaned.strip())

    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    return cleaned[start : end + 1]


def parse_design_output(raw_output):
    candidates = [clean_generated_text(raw_output)]
    extracted = extract_json_object(raw_output)
    if extracted is not None and extracted not in candidates:
        candidates.append(extracted)

    errors = []
    for candidate in candidates:
        try:
            return CausalDesignDraft.model_validate_json(candidate), candidate, errors
        except ValidationError as error:
            errors.append(error.errors()[0]['msg'])

    fallback_json = FALLBACK_RESPONSES['estimand_json']
    return CausalDesignDraft.model_validate_json(fallback_json), fallback_json, errors


parsed_design, parsed_json_used, parse_errors = parse_design_output(raw_structured_output)

if parse_errors:
    print('The live model did not return valid schema JSON on the first parse attempt.')
    print('Parser notes:', parse_errors)
    print('\nJSON used for validated object:')
    print(parsed_json_used)

parsed_design

The live model did not return valid schema JSON on the first parse attempt.
Parser notes: ['Invalid JSON: expected value at line 1 column 1']

JSON used for validated object:
{
 "treatment": "proactive outreach",
 "outcome": "churn",
 "unit": "customers",
 "time_window": "12 months",
 "main_assumptions": ["proactive outreach leads to reduced churn"],
 "missing_information": [],
 "unsafe_adjustment_variables": []
}


CausalDesignDraft(treatment='proactive outreach', outcome='churn', unit='customers', time_window='12 months', main_assumptions=['proactive outreach leads to reduced churn'], missing_information=[], unsafe_adjustment_variables=[])

### Discussion

The schema does not prove the causal design is correct. It only proves that the output has the expected structure. That is still useful because it creates a stable interface between the LLM and the rest of the analysis workflow.

When `RUN_LIVE_LOCAL_LLM = True`, a model may still return prose instead of JSON. That is not a notebook failure; it is an LLM reliability finding. The parser above first tries strict validation, then tries to extract a JSON object, then falls back to a deterministic example so the notebook can continue.

The next step is substantive validation: Are the assumptions right? Are any key confounders missing? Did the model hallucinate anything?

In [48]:
bad_output = {
    'treatment': 'outreach',
    'outcome': 'churn',
    # Missing unit, time_window, and main_assumptions.
}

try:
    CausalDesignDraft.model_validate(bad_output)
except ValidationError as error:
    print(error.errors()[:3])

[{'type': 'missing', 'loc': ('unit',), 'msg': 'Field required', 'input': {'treatment': 'outreach', 'outcome': 'churn'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('time_window',), 'msg': 'Field required', 'input': {'treatment': 'outreach', 'outcome': 'churn'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('main_assumptions',), 'msg': 'Field required', 'input': {'treatment': 'outreach', 'outcome': 'churn'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


## 10. Temperature and Reproducibility

For causal analysis, reproducibility matters. A stakeholder should not get a different causal design memo every time the notebook is run.

Use low temperature for artifacts that should be stable. Use higher temperature only for brainstorming, and always review the output.

In [49]:
temperature_guidance = pd.DataFrame(
    [
        ('0.0', 'deterministic or near-deterministic artifacts', 'estimand cards, variable-role tables, code skeletons'),
        ('0.1-0.3', 'controlled drafting', 'memos, summaries, diagnostic narratives'),
        ('0.5-0.8', 'brainstorming', 'possible mechanisms, sensitivity checks, alternative explanations'),
        ('>0.8', 'creative exploration', 'usually not appropriate for causal conclusions'),
    ],
    columns=['temperature', 'use', 'examples'],
)

temperature_guidance

,temperature,use,examples
0,0.0,deterministic or near-deterministic artifacts,"estimand cards, variable-role tables, code skeletons"
1,0.1-0.3,controlled drafting,"memos, summaries, diagnostic narratives"
2,0.5-0.8,brainstorming,"possible mechanisms, sensitivity checks, alternative explanations"
3,>0.8,creative exploration,usually not appropriate for causal conclusions


## 11. Hallucination in Causal Work

A hallucination is not only a fake citation. In causal inference, hallucination can mean inventing a design feature that would make the estimate credible if it were true.

Examples:

- claiming the treatment was randomized when it was not;
- claiming treatment timing is known when it is missing;
- claiming a variable is pre-treatment when it was measured after treatment;
- claiming overlap is adequate without checking it;
- claiming an instrument is valid without defending exclusion.

In [50]:
bad_causal_answer = causal_llm(
    'Summarize whether outreach reduced churn from a treated-versus-control comparison.',
    prompt_name='bad_answer',
)

high_risk_patterns = {
    'causal overclaim': ['caused', 'proves', 'impact', 'effectiveness', 'had an impact'],
    'unsupported design detail': ['randomized', 'instrument', 'threshold', 'before and after'],
    'unsupported inference detail': ['statistically significant', 'p-value', 'confidence interval'],
}

flags = [
    f'{category}: {term}'
    for category, terms in high_risk_patterns.items()
    for term in terms
    if term.lower() in bad_causal_answer.lower()
]

print(bad_causal_answer)
print('\nUnsupported high-risk terms:', flags)

To summarize, the effectiveness of outreach in reducing churn between treated and control groups can be assessed through statistical analysis or direct observation. If there is a significant reduction in churn rates observed after implementing outreach strategies, it suggests that the outreach was effective in decreasing churn. However, to make a definitive conclusion, additional data points such as changes in churn rates before and after outreach would be needed for a comprehensive evaluation.

Unsupported high-risk terms: ['causal overclaim: effectiveness', 'unsupported design detail: before and after']


### Discussion

This is a deliberately bad answer. A guardrail does not need to understand everything to be useful. Even a simple scan for high-risk terms can catch claims that require evidence.

In later notebooks, we will build richer evaluators that check outputs against structured project facts.

## 12. Evaluating LLM Outputs for Causal Quality

For this course, we will not evaluate model outputs by asking, "Did it sound smart?"

We will evaluate outputs using causal criteria.

In [51]:
causal_quality_rubric = pd.DataFrame(
    [
        ('estimand clarity', 'Does it define treatment, outcome, unit, population, and time window?'),
        ('association vs causation', 'Does it avoid causal claims from raw comparisons?'),
        ('confounder reasoning', 'Does it identify plausible common causes?'),
        ('bad-control detection', 'Does it avoid mediators, colliders, and post-treatment variables?'),
        ('assumption transparency', 'Does it state assumptions that cannot be tested directly?'),
        ('method fit', 'Does it recommend methods that match the assignment mechanism and data structure?'),
        ('evidence discipline', 'Does it avoid inventing randomization, instruments, cutoffs, or citations?'),
        ('decision usefulness', 'Does it communicate what decision can and cannot be supported?'),
    ],
    columns=['criterion', 'question'],
)

causal_quality_rubric

,criterion,question
0,estimand clarity,"Does it define treatment, outcome, unit, population, and time window?"
1,association vs causation,Does it avoid causal claims from raw comparisons?
2,confounder reasoning,Does it identify plausible common causes?
3,bad-control detection,"Does it avoid mediators, colliders, and post-treatment variables?"
4,assumption transparency,Does it state assumptions that cannot be tested directly?
5,method fit,Does it recommend methods that match the assignment mechanism and data structure?
6,evidence discipline,"Does it avoid inventing randomization, instruments, cutoffs, or citations?"
7,decision usefulness,Does it communicate what decision can and cannot be supported?


## 13. Model Scale and Model-Family Comparison Protocol

Because these are live local model calls, the exact ranking may change across model versions, package versions, seeds, tokenizer behavior, prompt wording, and reruns. The durable lesson is the evaluation pattern, not the leaderboard. Treat each table as an empirical snapshot of model behavior under this workflow.


With access to a high-memory GPU, we can compare every local model in a way that is directly relevant to causal analysis. The goal is not to decide which model sounds most polished. The goal is to see which model produces causal artifacts that are valid, cautious, structured, and useful.

The protocol is:

1. Give each model the same causal tasks.
2. Use the same system message.
3. Use low temperature for reproducibility.
4. Ask for both prose and structured artifacts.
5. Score each output using causal-quality checks.
6. Record failure modes, not only scores.

Set `RUN_FULL_MODEL_COMPARISON = True` in the configuration cell when you want to run the full benchmark interactively. It is intentionally `False` by default so the website renderer does not load every local model, including the 32B model, during a normal Quarto preview.

In [52]:
COMPARISON_SYSTEM_MESSAGE = textwrap.dedent(
    '''
    You are a careful causal inference assistant.
    Give final answers only; do not include scratch work.
    Distinguish association from causation.
    State assumptions explicitly.
    Do not invent randomization, instruments, thresholds, p-values, or confidence intervals.
    '''
).strip()

COMPARISON_STRUCTURED_PROMPT = textwrap.dedent(
    '''
    Draft a JSON object for the following causal design problem.

    Business question: Does proactive retention outreach reduce customer churn?
    Setting: Outreach was assigned by an internal churn-risk workflow, not by randomization.
    Candidate adjustment variables: prior churn risk score, prior support tickets,
    post-outreach engagement, and follow-up calls made after outreach.

    The JSON object must have exactly these keys:
    treatment, outcome, unit, time_window, main_assumptions,
    missing_information, unsafe_adjustment_variables.

    main_assumptions, missing_information, and unsafe_adjustment_variables must be arrays of strings.
    Return the JSON object only.
    '''
).strip()

COMPARISON_HIGH_RISK_TERMS = [
    'randomized',
    'proves',
    'caused',
    'impact',
    'effectiveness',
    'had an impact',
    'instrument',
    'threshold',
    'before and after',
    'statistically significant',
    'p-value',
    'confidence interval',
]


COMPARISON_TASKS = {
    'association_vs_causation': {
        'prompt': textwrap.dedent(
            '''
            Return exactly three concise bullet points explaining why an LLM should not conclude that
            a treatment caused an outcome from a raw treated-versus-control difference in observational data.
            '''
        ).strip(),
        'system_message': COMPARISON_SYSTEM_MESSAGE,
        'max_new_tokens': 220,
    },
    'estimand_json': {
        'prompt': COMPARISON_STRUCTURED_PROMPT,
        'system_message': JSON_ONLY_SYSTEM_MESSAGE,
        'max_new_tokens': 520,
    },
    'bad_control_audit': {
        'prompt': textwrap.dedent(
            '''
            We estimate the effect of proactive retention outreach on churn.
            Candidate controls are:
            - prior churn risk score
            - prior support tickets
            - post-outreach engagement
            - follow-up calls made after outreach

            Which variables are safe baseline controls, which are unsafe post-treatment controls,
            and why? Answer in no more than five bullets.
            '''
        ).strip(),
        'system_message': COMPARISON_SYSTEM_MESSAGE,
        'max_new_tokens': 280,
    },
    'executive_guardrail': {
        'prompt': textwrap.dedent(
            '''
            Given only this evidence: treated customers had 12% churn and untreated customers had 16% churn.
            Outreach was not randomized and targeted high-risk accounts. Write two sentences for an executive.
            Do not claim causality.
            '''
        ).strip(),
        'system_message': COMPARISON_SYSTEM_MESSAGE,
        'max_new_tokens': 180,
    },
}


def contains_any(text, terms):
    lowered = text.lower()
    return any(term in lowered for term in terms)


def count_bullets(text):
    return len(re.findall(r'(?m)^\s*(?:[-*]|\d+[.)])\s+', text))


def score_association_answer(text):
    checks = {
        'mentions confounding or selection': contains_any(text, ['confound', 'selection', 'self-select', 'baseline risk']),
        'mentions lack of randomization or assignment mechanism': contains_any(text, ['random', 'assignment', 'assigned']),
        'mentions timing or causal identification': contains_any(text, ['temporal', 'timing', 'identification', 'counterfactual', 'assumption']),
        'roughly follows three-point instruction': 2 <= count_bullets(text) <= 4,
        'avoids causal overclaim': not contains_any(text, ['proves', 'demonstrates causation', 'caused the outcome']),
    }
    return int(sum(checks.values())), checks


def score_design_json(raw_text):
    parsed, json_used, parse_errors = parse_design_output(raw_text)
    used_fallback = json_used == FALLBACK_RESPONSES['estimand_json']
    assumptions_text = ' '.join(parsed.main_assumptions).lower()
    missing_text = ' '.join(parsed.missing_information).lower()
    unsafe_text = ' '.join(parsed.unsafe_adjustment_variables).lower()

    checks = {
        'valid or extractable JSON': not used_fallback,
        'clear treatment/outcome/unit/time': all(
            [parsed.treatment, parsed.outcome, parsed.unit, parsed.time_window]
        ),
        'missing information is substantive': contains_any(
            missing_text,
            ['assignment', 'random', 'baseline', 'pre-treatment', 'timing', 'overlap', 'propensity'],
        ),
        'flags post-treatment controls': contains_any(
            unsafe_text,
            ['post', 'engagement', 'follow-up', 'followup', 'after outreach', 'mediator'],
        ),
        'assumptions are not causal conclusions': not contains_any(
            assumptions_text,
            ['outreach reduces churn', 'outreach leads to', 'proactive outreach leads'],
        ),
    }
    diagnostics = {
        'parsed': parsed.model_dump(),
        'json_used': json_used,
        'parse_errors': parse_errors,
    }
    return int(sum(checks.values())), checks, diagnostics


def score_bad_control_answer(text):
    checks = {
        'safe baseline controls identified': contains_any(text, ['prior churn risk', 'prior support tickets', 'baseline', 'pre-treatment']),
        'post-treatment variables identified': contains_any(text, ['post-outreach', 'after outreach', 'follow-up calls', 'engagement']),
        'explains mediator or bad-control risk': contains_any(text, ['mediator', 'post-treatment', 'bad control', 'part of the treatment effect']),
        'does not recommend controlling for everything': not contains_any(text, ['control for all', 'adjust for all', 'include all variables']),
    }
    return int(sum(checks.values())), checks


def score_executive_guardrail(text):
    checks = {
        'reports association descriptively': contains_any(text, ['treated', 'untreated', 'lower churn', 'difference', 'association']),
        'explicitly avoids causal claim': contains_any(text, ['cannot conclude', 'does not establish', 'not causal', 'not randomized', 'confounding']),
        'mentions assignment or targeting risk': contains_any(text, ['targeted', 'high-risk', 'selection', 'assignment', 'confounding']),
        'avoids unsupported inference details': not contains_any(text, COMPARISON_HIGH_RISK_TERMS),
    }
    return int(sum(checks.values())), checks


def score_model_outputs(outputs):
    association_score, association_checks = score_association_answer(outputs['association_vs_causation'])
    estimand_score, estimand_checks, estimand_diagnostics = score_design_json(outputs['estimand_json'])
    bad_control_score, bad_control_checks = score_bad_control_answer(outputs['bad_control_audit'])
    guardrail_score, guardrail_checks = score_executive_guardrail(outputs['executive_guardrail'])

    total_score = association_score + estimand_score + bad_control_score + guardrail_score
    max_score = 5 + 5 + 4 + 4

    diagnostics = {
        'association_vs_causation': association_checks,
        'estimand_json': estimand_checks,
        'estimand_json_details': estimand_diagnostics,
        'bad_control_audit': bad_control_checks,
        'executive_guardrail': guardrail_checks,
    }

    return {
        'association_score': association_score,
        'estimand_score': estimand_score,
        'bad_control_score': bad_control_score,
        'guardrail_score': guardrail_score,
        'total_score': total_score,
        'max_score': max_score,
        'score_share': total_score / max_score,
    }, diagnostics


def run_model_comparison(models_to_compare=MODELS_TO_COMPARE):
    if not RUN_LIVE_LOCAL_LLM:
        raise RuntimeError('Set RUN_LIVE_LOCAL_LLM = True before running the full model comparison.')

    rows = []
    artifacts = {}

    for label, model_id, role in models_to_compare:
        print(f'Running {label}: {model_id}')
        outputs = {}
        diagnostics = {}
        error = ''
        status = 'ok'

        try:
            for task_name, task in COMPARISON_TASKS.items():
                outputs[task_name] = local_chat(
                    task['prompt'],
                    system_message=task['system_message'],
                    model_id=model_id,
                    max_new_tokens=task.get('max_new_tokens', COMPARISON_MAX_NEW_TOKENS),
                    temperature=COMPARISON_TEMPERATURE,
                )
            empty_tasks = [task_name for task_name, output in outputs.items() if not clean_generated_text(output)]
            if empty_tasks:
                raise ValueError(f'Model returned empty output for tasks: {empty_tasks}')

            scores, diagnostics = score_model_outputs(outputs)
        except Exception as exc:
            status = 'error'
            error = repr(exc)[:500]
            scores = {
                'association_score': 0,
                'estimand_score': 0,
                'bad_control_score': 0,
                'guardrail_score': 0,
                'total_score': 0,
                'max_score': 18,
                'score_share': 0.0,
            }
        finally:
            artifacts[label] = {
                'model_id': model_id,
                'role': role,
                'outputs': outputs,
                'diagnostics': diagnostics,
                'error': error,
            }
            clear_loaded_model_cache()

        rows.append(
            {
                'label': label,
                'model_id': model_id,
                'role': role,
                'status': status,
                **scores,
                'error': error,
            }
        )

    results = pd.DataFrame(rows)
    results['_status_order'] = results['status'].map({'ok': 0, 'error': 1}).fillna(2)
    results = results.sort_values(['_status_order', 'score_share'], ascending=[True, False])
    return results.drop(columns='_status_order'), artifacts


if RUN_FULL_MODEL_COMPARISON:
    model_comparison_results, model_comparison_outputs = run_model_comparison()
else:
    model_comparison_results = pd.DataFrame(MODELS_TO_COMPARE, columns=['label', 'model_id', 'role'])
    model_comparison_results['status'] = 'not run'
    for column in [
        'association_score',
        'estimand_score',
        'bad_control_score',
        'guardrail_score',
        'total_score',
        'max_score',
        'score_share',
    ]:
        model_comparison_results[column] = np.nan
    model_comparison_results['error'] = ''
    model_comparison_outputs = {}
    display(
        Markdown(
            'Full model comparison is ready but not executed. '
            'Set `RUN_FULL_MODEL_COMPARISON = True` in the configuration cell, then rerun this section in the GPU notebook session.'
        )
    )



def build_comparison_diagnostics_table(model_outputs):
    rows = []
    for label, artifact in model_outputs.items():
        diagnostics = artifact.get('diagnostics', {})
        for task_name, task_checks in diagnostics.items():
            if task_name.endswith('_details'):
                continue
            for check, passed in task_checks.items():
                rows.append(
                    {
                        'label': label,
                        'task': task_name,
                        'check': check,
                        'passed': bool(passed),
                    }
                )
    return pd.DataFrame(rows)


def build_output_preview_table(model_outputs, characters=420):
    rows = []
    for label, artifact in model_outputs.items():
        for task_name, output in artifact.get('outputs', {}).items():
            rows.append(
                {
                    'label': label,
                    'task': task_name,
                    'output_preview': clean_generated_text(output)[:characters],
                }
            )
    return pd.DataFrame(rows)


if RUN_FULL_MODEL_COMPARISON:
    completed = model_comparison_results[model_comparison_results['status'].eq('ok')].copy()
    comparison_diagnostics_table = build_comparison_diagnostics_table(model_comparison_outputs)
    model_output_preview_table = build_output_preview_table(model_comparison_outputs)

    if not completed.empty:
        best = completed.sort_values('score_share', ascending=False).iloc[0]
        display(
            Markdown(
                f"Best model by this causal-quality rubric: **{best['label']}** "
                f"with {best['total_score']:.0f}/{best['max_score']:.0f} checks passed."
            )
        )
    display(comparison_diagnostics_table)
    display(model_output_preview_table)
else:
    comparison_diagnostics_table = pd.DataFrame(columns=['label', 'task', 'check', 'passed'])
    model_output_preview_table = pd.DataFrame(columns=['label', 'task', 'output_preview'])

model_comparison_results


Running Qwen 0.5B: Qwen/Qwen2.5-0.5B-Instruct
Running Qwen 7B: Qwen/Qwen2.5-7B-Instruct


Running Qwen 14B: Qwen/Qwen2.5-14B-Instruct


Running Qwen 32B: Qwen/Qwen2.5-32B-Instruct


Running Phi mini: microsoft/Phi-3.5-mini-instruct


Running Mistral 7B: mistralai/Mistral-7B-Instruct-v0.3


...

Running Llama 3.1 8B: meta-llama/Meta-Llama-3.1-8B-Instruct


Best model by this causal-quality rubric: **Qwen 7B** with 16/18 checks passed.

,label,task,check,passed
0,Qwen 0.5B,association_vs_causation,mentions confounding or selection,False
1,Qwen 0.5B,association_vs_causation,mentions lack of randomization or assignment mechanism,False
2,Qwen 0.5B,association_vs_causation,mentions timing or causal identification,False
3,Qwen 0.5B,association_vs_causation,roughly follows three-point instruction,True
4,Qwen 0.5B,association_vs_causation,avoids causal overclaim,True
...,...,...,...,...
157,Llama 3.1 8B,bad_control_audit,does not recommend controlling for everything,True
158,Llama 3.1 8B,executive_guardrail,reports association descriptively,True
159,Llama 3.1 8B,executive_guardrail,explicitly avoids causal claim,False
160,Llama 3.1 8B,executive_guardrail,mentions assignment or targeting risk,False


,label,task,output_preview
0,Qwen 0.5B,association_vs_causation,1. An LLM cannot definitively state causation without additional evidence.\n2. Observational studies lack control gr...
1,Qwen 0.5B,estimand_json,"```json\n{\n ""treatment"": ""proactive retention outreach"",\n ""outcome"": ""customer churn"",\n ""unit"": [""churn risk scor..."
2,Qwen 0.5B,bad_control_audit,Safety Baseline Controls:\n\n1. Prior churn risk score: This is a measure that reflects an individual's likelihood o...
3,Qwen 0.5B,executive_guardrail,Treated customers experienced lower churn rates compared to untreated customers due to outreach efforts that were no...
4,Qwen 7B,association_vs_causation,- An LLM should not conclude causation from a treated-versus-control difference in observational data because unobse...
5,Qwen 7B,estimand_json,"{\n ""treatment"": ""proactive retention outreach"",\n ""outcome"": ""customer churn"",\n ""unit"": ""customer"",\n ""time_window..."
6,Qwen 7B,bad_control_audit,- **Prior churn risk score**: Safe baseline control. This variable measures risk before any intervention and is not ...
7,Qwen 7B,executive_guardrail,"Treated customers showed a lower churn rate at 12% compared to 16% for untreated customers, but this observation doe..."
8,Qwen 14B,association_vs_causation,"- Observational data lacks random assignment, making it impossible to rule out confounding variables that affect bot..."
9,Qwen 14B,estimand_json,"{\n ""treatment"": ""proactive retention outreach"",\n ""outcome"": ""customer churn"",\n ""unit"": ""customer"",\n ""time_window..."


,label,model_id,role,status,association_score,estimand_score,bad_control_score,guardrail_score,total_score,max_score,score_share,error
1,Qwen 7B,Qwen/Qwen2.5-7B-Instruct,fast default,ok,4,5,4,3,16,18,0.889,
2,Qwen 14B,Qwen/Qwen2.5-14B-Instruct,strong local analysis,ok,4,4,4,3,15,18,0.833,
3,Qwen 32B,Qwen/Qwen2.5-32B-Instruct,scale comparison,ok,4,4,4,3,15,18,0.833,
4,Phi mini,microsoft/Phi-3.5-mini-instruct,compact non-Qwen comparison,ok,5,4,4,2,15,18,0.833,
6,Mistral Small 24B,mistralai/Mistral-Small-3.1-24B-Instruct-2503,strong non-Qwen comparison,ok,4,4,4,3,15,18,0.833,
7,Gemma 3 27B,google/gemma-3-27b-it,large non-Qwen comparison,ok,3,4,4,3,14,18,0.778,
8,Llama 3.1 8B,meta-llama/Meta-Llama-3.1-8B-Instruct,industry-standard instruct baseline,ok,2,5,4,2,13,18,0.722,
0,Qwen 0.5B,Qwen/Qwen2.5-0.5B-Instruct,pipeline smoke test,ok,2,2,4,3,11,18,0.611,
5,Mistral 7B,mistralai/Mistral-7B-Instruct-v0.3,7B model-family comparison,ok,2,2,4,1,9,18,0.500,


## 14. Human-in-the-Loop Pattern

A safe LLM workflow has three layers:

1. **Generate:** the model drafts an artifact.
2. **Validate:** code checks structure and known facts.
3. **Review:** a causal analyst checks assumptions and interpretation.

The human review is not decorative. It is part of the identification strategy.

In [53]:
review_checklist = pd.DataFrame(
    [
        ('Does the output define the estimand?', False),
        ('Does the output separate association from causation?', False),
        ('Does the output list required assumptions?', False),
        ('Does the output avoid post-treatment adjustment?', False),
        ('Does the output flag missing timing or assignment information?', False),
        ('Does the output avoid invented design facts?', False),
        ('Does the output state limitations in stakeholder-ready language?', False),
    ],
    columns=['review question', 'analyst approved'],
)

review_checklist

,review question,analyst approved
0,Does the output define the estimand?,False
1,Does the output separate association from causation?,False
2,Does the output list required assumptions?,False
3,Does the output avoid post-treatment adjustment?,False
4,Does the output flag missing timing or assignment information?,False
5,Does the output avoid invented design facts?,False
6,Does the output state limitations in stakeholder-ready language?,False


## 15. Practical Rules for This Course

We will use these rules in the remaining notebooks:

- Use deterministic fallback outputs for public rendering.
- Use local LLMs interactively for real demos and model comparisons.
- Use chat templates for local instruction models.
- Use low temperature for causal artifacts.
- Use structured outputs when outputs feed later code.
- Score model outputs with causal criteria.
- Treat model scale and model family as empirical questions.
- Never let an LLM turn observational association into a causal conclusion without design evidence.

## 16. Key Takeaways

- LLMs are useful causal-analysis copilots, but they are not causal authorities.
- Prompt format matters. For local chat models, use the tokenizer's chat template when available.
- Low temperature and explicit seeds support reproducibility.
- Structured outputs make LLM artifacts easier to validate.
- Hallucination in causal work often means inventing design credibility: randomization, timing, instruments, or valid adjustment.
- Model scale is useful, but bigger models can still make causal mistakes.
- Model-family comparisons are useful when we score outputs with causal criteria instead of general impressions.

The next notebook turns vague business questions into precise causal questions using these LLM workflow patterns.